**Quickstart QC: the registrations of step 2 and the review table**
The QC of the runs of `1.quickstart_direct_normalize.ipynb`, apart from the pipeline: nothing here changes or holds up a run there, and a failed or flagged QC does not stop the next step. There is no QC of step 1: errors of its brain mask show in the QC of the registrations. It can start while step 2 is still running: a run needs only its own registrations. The review table written here chooses what step 3 (`2.full_pipeline.ipynb`) uses.

| section | function | what | output |
|---|---|---|---|
| 2 | `direct_norm_qc` | per run the reference cerebellum (`brain_to_cerebellum`: SUITPy on the N4 T1 with the T2 and the brain mask, 2-3 runs, the medoid; chosen without any registration); per template: T1 r, brain and cerebellum Dice and HD95, Jacobian | `<prefix>_direct_normalize_qc.json`, `<prefix>_direct_to-<template>_qc.png`; `direct_review.tsv` |

Decisions after looking at the figures are written into the dictionary of the review cell (`DIRECT_DECISIONS`), so the table can be made again at any time. Sections: 0 setup, 1 the dataset table, 2 QC of steps 1-2, 3 what goes to step 3.

# 0. Setup
Installation (once per computer): 0. Setup of `1.quickstart_direct_normalize.ipynb`; kernel `<conda>/envs/neonorm/bin/python`.

**What you fill in** (the rest runs as it is): the block `FILL IN` of the cell below; `V` in 2.1; `SUBMIT_STEP2_QC = True` in 2.2 to submit; `SAVE_FIG` in 2.3 (optional); `DIRECT_DECISIONS` in 2.5.


In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import display

# ===== FILL IN: the same values as in 1.quickstart_direct_normalize.ipynb =====
ENV_FILE = Path('/project/4290000.01/yapwan/toolbox/neonorm_test/neonorm_env.sh')  # written by setup_neonorm.sh
EXAMPLE = Path('example')    # the folder of participants.tsv; relative to docs/tutorials
DATA_ROOT = EXAMPLE / 'out'  # the outputs: one folder per visit, the review tables, the Slurm logs
N_THREADS = 8                # ANTs threads in this notebook (a Slurm job uses the CPUs it asks for)
# ==============================================================================

SETUP = f'source {ENV_FILE}'  # the setup line of the Slurm job scripts
assert ENV_FILE.exists(), f'ENV_FILE not found: {ENV_FILE}'
# the settings of ENV_FILE in this notebook
env = subprocess.run(['bash', '-c', f'source {ENV_FILE} && env -0'], capture_output=True, check=True).stdout
os.environ.update(kv.split('=', 1) for kv in env.decode().split('\0') if '=' in kv)
os.environ['NEONORM_NTHREADS'] = str(N_THREADS)
# work in <repository>/docs/tutorials (the kernel may start in another folder)
nb_file = globals().get('__vsc_ipynb_file__')
REPO = Path(nb_file).resolve().parents[2] if nb_file else Path(os.environ['NEONORM_BCP_TEMPLATES']).parents[1]
os.chdir(REPO / 'docs' / 'tutorials')
EXAMPLE, DATA_ROOT = EXAMPLE.resolve(), DATA_ROOT.resolve()  # absolute: the jsons record the paths
sys.path.insert(0, str(REPO / 'Script'))
import neonorm

problems = neonorm.check_environment(verbose=True)
print('\nready' if not problems else '\nnot ready:\n  - ' + '\n  - '.join(problems))


# 1. The dataset
The table of `1.quickstart_direct_normalize.ipynb`, section 3.1.

In [ ]:
visits = pd.read_csv(EXAMPLE / 'participants.tsv', sep='\t', keep_default_na=False)
if 'pma_weeks' in visits:  # postmenstrual age in weeks instead of age_months
    visits['age_months'] = neonorm.age_months_from_pma(visits['pma_weeks'].astype(float))
if 'brain_mask' not in visits:
    visits['brain_mask'] = ''
for c in ('t1', 't2', 'brain_mask'):  # absolute paths (relative ones are relative to the table)
    visits[c] = [str(EXAMPLE / f) if f else '' for f in visits[c]]
visits['out_dir'] = [str(DATA_ROOT / i) for i in visits['participant_id']]
assert visits['participant_id'].is_unique
DATA_ROOT.mkdir(parents=True, exist_ok=True)
visits

# 2. QC of steps 1-2 (`direct_norm_qc`)
`direct_norm_qc(normalize_json)`, apart from `direct_normalize`; command line `neonorm.py --direct_norm_qc <prefix>_direct_normalize.json`. Per run (`<prefix>_direct_normalize.json` in the visit folder): the reference cerebellum of the visit, then the QC of every registration. **Resources**: 4 CPUs, 24 GB (SUITPy).

## 2.1 One visit (here in Python)

In [ ]:
def step2_jsons(v):
    """The <prefix>_direct_normalize.json of the step 2 runs of a visit (its folder)."""
    return sorted(Path(v.out_dir).glob('*_direct_normalize.json'))


# V = visits.iloc[0]
# for nj in step2_jsons(V):
#     q = neonorm.direct_norm_qc(nj)
#     print(nj.name)
#     display(pd.DataFrame(q['qc']).T[['t1_corr', 'brain_dice', 'brain_log10_hd95', 'cerebellum_dice',
#                                      'cerebellum_log10_hd95', 'jacobian_min']])

## 2.2 All runs (Slurm)
Every run whose registrations are all done (all templates of `direct_templates(age)` in its json) and that has no QC yet (`REDO_STEP2_QC = True`: all). `submit_direct_normalize_array(..., register=False, qc=True)` submits one job array `directqc` (one task per run); the arguments come from the json of the run.

In [ ]:
def qc_args(nj):
    """Arguments of direct_normalize_job_scripts for the QC of a run, from its <prefix>_direct_normalize.json."""
    log = json.loads(nj.read_text())
    i = log['inputs']
    return {'t1_brain': log['head2brain']['t1_brain'], 'age_months': i['age_months'], 'out_dir': str(nj.parent),
            'prefix': i['prefix'], 'templates': list(log.get('normalize', {}))}


REDO_STEP2_QC = False
SUBMIT_STEP2_QC = True  # True: submit
todo, waiting = [], []
for v in visits.itertuples():
    for nj in step2_jsons(v):
        a = qc_args(nj)
        if not set(neonorm.direct_templates(v.age_months)) <= set(a['templates']):
            waiting.append(a['prefix'])
        elif REDO_STEP2_QC or not (nj.parent / f"{a['prefix']}_direct_normalize_qc.json").exists():
            todo.append(a)
print(f'{len(todo)} runs to QC; registrations not done yet: {waiting}')
if SUBMIT_STEP2_QC and todo:
    arr = neonorm.submit_direct_normalize_array(todo, DATA_ROOT / 'logs' / datetime.now().strftime('step2qc_%Y%m%d_%H%M%S'),
                                                register=False, qc=True, setup=SETUP, max_running=50)
    print('array', arr['qc_job_id'])

## 2.3 The metrics by age, against the reference
One row per run and template from `<prefix>_direct_normalize_qc.json` (`role`: `nearest` = the main template, `MNI`, `other` = the other BCP templates of the age window), flagged by the rules below (`neonorm.direct_qc_flags`). The figure (`neonorm.plot_direct_qc_by_age`): the QC metrics by age, to the nearest BCP template (left) and to MNI (right); x: the age groups of the BCP templates (0M-60M), then 8-18Y, each registration at its real age. `SAVE_FIG`: optional, a file for the figure (e.g. `.png` or `.pdf`). Below the figure: every flagged registration with its reasons (each flag: the value, the median of the reference, z and which direction is worse).
- **Ref** (grey): the registrations of this project that passed the visual QC, the same ones whose median and scale the flags use (`Script/data/direct_qc_reference.tsv`). To the nearest BCP template: BCP (0-81 months) and HCPD (8-18 years) visits, direct normalisation of notebook 3 (3.3, `avgsubj_usable = yes`; 811 registrations). To MNI: the direct registrations to MNI of the same datasets, notebook 5 (5.1, `direct_usable = yes`; 783 registrations). Values in `Script/data/direct_qc_reference_values.tsv`: age (0.1 month) and the metrics, no IDs (made by `Script/make_direct_qc_reference.py` of the project).
- **flagged** (red, with its prefix; in the panel of the check that flagged it also the reason, z or T1 r < 0.5): a new registration flagged; **ok** (blue): a new registration not flagged.

**Flag rules**: A flag means: look at the QC figure of this registration, not that it failed. Each registration is compared with the registrations of the project (Ref, above) to the same template (MNI by age group: 0-3, 3-6, 6-12, 12-24, 24-96, 96+ months; `Script/data/direct_qc_reference.tsv`). The reference is frozen, so a flag does not depend on the other visits of this dataset. Flagged if:
- an outlier: robust z = (value - median of Ref) / scale of Ref, beyond 3 in the worse direction: T1 r z < -3 (lower = worse), brain or cerebellum log10 HD95 z > 3 (higher = worse); the HD95 also at least 2.4 mm (BCP) / 3 mm (MNI) above the median of the reference (3 voxels); a missing value is flagged;
- MNI: T1 r < 0.5 (in the project T1 r with MNI has no value between 0.26 and 0.50: below it the registration likely failed; direct registration to MNI usually fails in the first months).

Not flag checks, but also worth a look: points far from Ref in the figure for Dice or folding. Many flags in one direction across this dataset point to a difference of the data (scanner, population, e.g. preterm infants at term-equivalent age), not to failed registrations: the decision is made on the figure (2.5).


In [ ]:
SAVE_FIG = None  # optional: a file for the figure, e.g. DATA_ROOT / 'direct_qc_by_age.png'
METRICS = ['t1_corr', 'brain_dice', 'brain_hd95', 'brain_log10_hd95', 'cerebellum_dice', 'cerebellum_hd95',
           'cerebellum_log10_hd95', 'jacobian_min']
rows = []
for v in visits.itertuples():
    for nj in step2_jsons(v):
        f = nj.with_name(nj.name.replace('.json', '_qc.json'))
        if not f.exists():
            continue
        q = json.loads(f.read_text())
        prefix = json.loads(nj.read_text())['inputs']['prefix']
        for tpl, m in q['qc'].items():
            rows.append({'participant_id': v.participant_id, 'prefix': prefix, 'age_months': v.age_months,
                         'template': tpl,
                         'role': 'nearest' if tpl == q['main_template'] else ('MNI' if tpl == neonorm.MNI_NAME else 'other'),
                         **{k: m.get(k) for k in METRICS},
                         'qc_figure': m.get('qc_figure')})
direct_qc = pd.DataFrame(rows)
if len(direct_qc):
    direct_qc = neonorm.direct_qc_flags(direct_qc)
    print(f"{len(direct_qc)} registrations of {direct_qc['prefix'].nunique()} runs, {direct_qc['flagged'].sum()} registrations flagged")
    fig = neonorm.plot_direct_qc_by_age(direct_qc)
    if SAVE_FIG:
        fig.savefig(SAVE_FIG, dpi=300)
    plt.show()
    # the reason of every flagged registration (z beyond 3 in the worse direction)
    for r in direct_qc[direct_qc['flagged']].sort_values(['age_months', 'prefix']).itertuples():
        print(f'{r.prefix} -> {r.template} ({r.role}), {r.age_months:.2f} months:')
        for flag in r.flags.split(' | '):
            print(f'    - {flag}')


## 2.4 The flagged registrations
To look at before the review (2.5): the QC figure of each flagged registration. Template above, warped T1 below; template brain mask red, cerebellum green. The line on top: the registration and its metrics; in red the metrics that flagged it, with z and which direction is worse (details in 2.3).

In [ ]:
import re

WORSE = {'t1_corr': 'lower = worse', 'brain_log10_hd95': 'higher = worse', 'cerebellum_log10_hd95': 'higher = worse'}


def header(r):
    """The metrics line of a QC figure as (text, red) parts: a metric that flagged the registration red, with its reason."""
    def reason(k):
        why = neonorm._flag_reason(r.flags or '', k, getattr(r, f'z_{k}'))
        return re.sub(r'(z [-+][\d.]+)', rf'\1, {WORSE[k]}', why)
    parts = [(f'{r.prefix} -> {r.template} ({r.role}, {r.age_months:.2f} months)', ''),
             (f'T1 r {r.t1_corr:.3f}', reason('t1_corr')),
             (f'brain Dice {r.brain_dice:.3f}, HD95 {r.brain_hd95:.1f} mm', reason('brain_log10_hd95')),
             (f'cerebellum Dice {r.cerebellum_dice:.3f}, HD95 {r.cerebellum_hd95:.1f} mm',
              reason('cerebellum_log10_hd95')),
             (f'min Jacobian {r.jacobian_min:.2f}', '')]
    return [(text + why, bool(why)) for text, why in parts]


for r in direct_qc[direct_qc['flagged']].sort_values(['age_months', 'prefix']).itertuples():
    lines, n = [[]], 0  # the parts in lines of about 140 characters
    for text, red in header(r):
        if lines[-1] and n + len(text) > 140:
            lines.append([])
            n = 0
        lines[-1].append((text, red))
        n += len(text) + 3
    img = plt.imread(r.qc_figure)
    ink = (img[..., :3] < 0.8).any(axis=(1, 2))
    start = ink.argmax()
    img = img[start + ink[start:].argmin():]  # without the metrics line of the QC figure (written again above)
    h, w = img.shape[:2]
    top = 0.025 * h * (len(lines) + 0.5)
    fig = plt.figure(figsize=(9, 9 * (h + top) / w))
    ax = fig.add_axes((0, 0, 1, h / (h + top)))
    ax.imshow(img)
    ax.axis('off')
    hdr = fig.add_axes((0, h / (h + top), 1, top / (h + top)))
    hdr.axis('off')
    for j, line in enumerate(lines):
        prev = None
        for i, (text, red) in enumerate(line):
            text += ' | ' if j < len(lines) - 1 or i < len(line) - 1 else ''
            kw = dict(fontsize=6.5, color=neonorm.WAY_COLORS['avgsubj'] if red else 'black', va='bottom')
            prev = (hdr.text(0.02, 1 - (j + 1) / (len(lines) + 0.5), text, **kw) if prev is None else
                    hdr.annotate(text, xy=(1, 0), xycoords=prev, **kw))
    plt.show()

## 2.5 Review → `direct_usable` per registration
**How to review**
1. Look at the QC figures of 2.4 (the flagged registrations) and, if you like, of other registrations far from Ref in 2.3 (`<prefix>_direct_to-<template>_qc.png` in the visit folder).
2. Decide on each figure with the rules below. Most flagged registrations are fine: a flag only asks for a look.
3. Write every `'no'` into `DIRECT_DECISIONS` in the cell below, with the rule as a comment; all other registrations stay `yes`. Key: `(prefix, template)` for one registration, or `prefix` alone for all registrations of a run (an image problem, rule 2).
4. Run the cell: it writes `<DATA_ROOT>/direct_review.tsv` (every registration with `direct_usable`), which step 3 reads. The decisions stay in the dictionary, so the table can be made again at any time; when new visits are QC'd, add their decisions and run it again.

**Rules**
1. Only this registration is off, the other registrations of the visit are fine -> judge the registration; `'no'` (failed) if:
   - **R1 misaligned**: the warped brain or cerebellum does not lie on the outlines (shifted, rotated, mirrored, sheared or scaled; about 1 cm or more);
   - **R2 boundary**: brain tissue (gyri) outside the red outline, or a part of it not filled, by more than about 3 voxels; not counted: extra-axial CSF or dura outside the outline, the empty vertex rim of infants;
   - **R3 inner structures**: ventricles, corpus callosum, thalami or brainstem misplaced while the outlines fit;
   - **R4 deformation**: folds or tears (minimum Jacobian <= 0);
   otherwise `yes`, also when a value is low.
2. All registrations of the visit are off in the same way -> judge the image (key: the prefix alone): **I1** motion artefacts over much of the brain, **I2** image incomplete (cut field of view, signal dropout), **I3** brain mask cuts off brain or keeps large non-brain parts -> `'no'`; **I4** weak or inverted contrast (newborn), unusual anatomy with the structures in place -> `yes`.

**What a `'no'` means for step 3**: `direct` uses the MNI registration, `tpl2tpl` and `avgsubj` the registration to the nearest BCP template; a way whose registration is `no` is left out for that visit.


In [ ]:
DIRECT_DECISIONS = {  # (prefix, template) or prefix (all its registrations): 'no'; the rule as a comment, e.g.
    # ('sub-ex01', 'MNI152NLin2009cSym'): 'no',  # R1 misaligned
    # 'sub-xx': 'no',                            # I1 motion, all registrations
    ('sub-ex01', 'MNI152NLin2009cSym'): 'no',  # R1 misaligned
    ('sub-ex01', 'BCP00M'): 'yes',
    ('sub-ex01', 'BCP01M'): 'yes',
    ('sub-ex01', 'BCP02M'): 'yes',
}
unknown = [k for k in DIRECT_DECISIONS
           if not ((direct_qc['prefix'] == k[0]) & (direct_qc['template'] == k[1]) if isinstance(k, tuple)
                   else direct_qc['prefix'] == k).any()]
assert not unknown, f'decisions for no registration of the table: {unknown}'
direct_qc['direct_usable'] = [DIRECT_DECISIONS.get((p, t), DIRECT_DECISIONS.get(p, 'yes'))
                              for p, t in zip(direct_qc['prefix'], direct_qc['template'])]
direct_qc.to_csv(DATA_ROOT / 'direct_review.tsv', sep='\t', index=False)
print(f"direct_review.tsv: {len(direct_qc)} registrations, {(direct_qc['direct_usable'] != 'yes').sum()} not usable")
direct_qc.loc[direct_qc['flagged'] | (direct_qc['direct_usable'] != 'yes'),
              ['prefix', 'template', 'role', 'age_months', 'flags', 'direct_usable']]


# 3. What goes to step 3
`2.full_pipeline.ipynb` reads `direct_review.tsv`: per run only the registrations with `direct_usable == yes`. The reference cerebellum `<prefix>_cerebellum_suitpy_from-brain-N4_dseg.nii.gz` made here (2.1-2.2) is the cerebellum moved to MNI in step 3.